In [ ]:
import numpy as np 
import pandas as pd 
import os
import math
import optuna
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import time
from tqdm import tqdm
tqdm.pandas()
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_val_score, RandomizedSearchCV


# Imports for Deep Learning
from keras.layers import Conv2D, Dense, Dropout, Flatten
from keras.models import Sequential


# ensure consistency across runs
from numpy.random import seed
seed(1)

#set_random_seed(2)

# Imports to view data
import cv2
from glob import glob
from matplotlib import pyplot as plt
from numpy import floor
import random


In [ ]:
train = pd.read_csv("/kaggle/input/zindi-c/Train.csv")
test = pd.read_csv('/kaggle/input/zindi-c/Test.csv')
sub = pd.read_csv("/kaggle/input/zindi-c/SampleSubmission.csv")
train

In [ ]:
print(train.shape)
train.describe()

# Simple EDA

In [ ]:
sns.histplot(train["GT_NO2"], bins=202, kde=False, color='blue', edgecolor='black')

# Customize labels and title
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Histogram of GT_No2')

# Display the plot
plt.show()

In [ ]:
sns.histplot(train["TropopausePressure"], bins=9, kde=False, color='blue', edgecolor='black')

# Customize labels and title
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Histogram of TropopausePressure')

# Display the plot
plt.show()

In [ ]:
# Create a figure and a set of subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot the first histogram in the first subplot
sns.histplot(train["CloudFraction"], bins=50, kde=False, color='blue', edgecolor='black', ax=axes[0, 0])
axes[0, 0].set_xlabel('Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Histogram of CloudFraction')

# Plot the second histogram in the second subplot
sns.histplot(train["CloudFraction"], bins=50, kde=False, color='blue', edgecolor='black', ax=axes[0, 1])
axes[0, 1].set_xlabel('Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Histogram of CloudFraction')

# Plot the third histogram in the third subplot
sns.histplot(train["NO2_strat"], bins=200, kde=False, color='blue', edgecolor='black', ax=axes[0, 2])
axes[0, 2].set_xlabel('Value')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].set_title('Histogram of NO2_strat')

# Plot the fourth histogram in the fourth subplot
sns.histplot(train["NO2_total"], bins=50, kde=False, color='blue', edgecolor='black', ax=axes[1, 0])
axes[1, 0].set_xlabel('Value')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Histogram of NO2_total')

# Plot the fifth histogram in the fifth subplot
sns.histplot(train["NO2_trop"], bins=50, kde=False, color='blue', edgecolor='black', ax=axes[1, 1])
axes[1, 1].set_xlabel('Value')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Histogram of NO2_trop')

# Plot the sixth histogram in the sixth subplot
sns.histplot(train["LST"], bins=500, kde=False, color='blue', edgecolor='black', ax=axes[1, 2])
axes[1, 2].set_xlabel('Value')
axes[1, 2].set_ylabel('Frequency')
axes[1, 2].set_title('Histogram of LST')

# Adjust layout
plt.tight_layout()

# Display the plot
plt.show()

## Let's use the date feature for a plot

In [ ]:
train['Date'] = pd.to_datetime(train['Date'])

In [ ]:
plt.figure(figsize=(12, 6))

# Plot using Seaborn's lineplot
sns.lineplot(x = train["Date"], y="GT_NO2", data=train)

# Rotate x-axis labels for better readability (optional)
plt.xticks(rotation=45)

# Add labels and title
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Time Series Plot')

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
# Plot using Seaborn's lineplot
sns.lineplot(x = train["Date"], y="LST", data=train)

# Rotate x-axis labels for better readability (optional)
plt.xticks(rotation=45)

# Add labels and title
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Time Series Plot')

# Display the plot
plt.tight_layout()
plt.show()

# Now some Preprocessing

In [ ]:
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

def Prep(df):
    df['Date'] = pd.to_datetime(df['Date'])
    
    numeric_columns = df.select_dtypes(include=['number']).columns
    
    for col in numeric_columns:
        # Fill missing values temporarily using linear interpolation
        data_interpolated = df[col].interpolate(method='linear')
        
        # Handle cases where interpolation might still leave NaNs at the ends
        if data_interpolated.isna().sum() > 0:
            data_interpolated.fillna(method='bfill', inplace=True)
            data_interpolated.fillna(method='ffill', inplace=True)

        # Decompose the time series to extract the trend component
        decomposition = seasonal_decompose(data_interpolated, model='additive', period=30)
        trend = decomposition.trend
        
        # Handle cases where the trend might still have NaNs at the ends
        trend.fillna(method='bfill', inplace=True)
        trend.fillna(method='ffill', inplace=True)

        # Replace original NaN values with the trend component
        df[col] = df[col].combine_first(trend)
        
        # Fill any remaining NaN values with the mean of the column
        df[col].fillna(value=df[col].mean(), inplace=True)
    
    # Define the rolling window size (e.g., 3 days)
    window_size = 3

    # Exclude the 'Date' column from rolling calculations
    rolling_numeric_columns = df[numeric_columns]
    
    # Calculate rolling mean for each numeric feature
    df_rolling_mean = rolling_numeric_columns.rolling(window=window_size).mean()

    # Calculate rolling standard deviation for each numeric feature
    df_rolling_std = rolling_numeric_columns.rolling(window=window_size).std()

    # Rename the columns to indicate they are rolling statistics
    df_rolling_mean.columns = [f'{col}_rolling_mean' for col in df_rolling_mean.columns]
    df_rolling_std.columns = [f'{col}_rolling_std' for col in df_rolling_std.columns]

    # Concatenate the rolling statistics with the original dataframe
    df_combined = pd.concat([df, df_rolling_mean, df_rolling_std], axis=1)

    # Extract year, month, and day into separate columns
    df_combined['year'] = df_combined['Date'].dt.year
    df_combined['month'] = df_combined['Date'].dt.month
    df_combined['day'] = df_combined['Date'].dt.day

    # Additional feature engineering
    if 'NO2_trop' in df_combined.columns and 'NO2_strat' in df_combined.columns:
        df_combined["NO2_Ratio"] = df_combined["NO2_trop"] / df_combined["NO2_strat"]
    else:
        df_combined["NO2_Ratio"] = 0
        
    if 'NO2_strat' in df_combined.columns and 'NO2_total' in df_combined.columns and 'NO2_trop' in df_combined.columns:
        df_combined["SUM"] = df_combined["NO2_strat"] + df_combined["NO2_total"] + df_combined["NO2_trop"]
    else:
        df_combined["SUM"] = 0
        
    if 'LST' in df_combined.columns:
        df_combined["LSTsqr"] = df_combined["LST"] ** 2
    else:
        df_combined["LSTsqr"] = 0
        
    if 'Precipitation' in df_combined.columns:
        df_combined["Precipitationsqr"] = df_combined["Precipitation"] ** 2
    else:
        df_combined["Precipitationsqr"] = 0
    
    df_combined.fillna(value = 0, inplace=True)
    
    return df_combined

# Assuming `train` and `test` are your DataFrames
train_prepped = Prep(train)
test_prepped = Prep(test)


### The modeling wasn't published here if this notebook get some interest and feedback I might publish the modeling and optimization and extra plots in a future notebook 
$$I \ hope \ this \ help \ you \ if \ you \ have \ questions \ or \ feedback \ just \ comment$$